In [1]:
import time
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

#### <span style="background-color: yellow;">1. INITIALISATION & FONCTION DE CHRONO</span>

In [2]:
spark = SparkSession.builder \
    .appName("Benchmark_Optimisation_Spark") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "1g") \
    .config("spark.executor.cores", "1") \
    .getOrCreate()

In [3]:
# On désactive l'AutoBroadcast dès le départ pour observer la performance d'un Join classique (SortMergeJoin)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [4]:
results = {}
def mesurer_requete(nom_etape, fonction_requete):
    print(f"--- Exécution : {nom_etape} ---")
    debut = time.time()
    df_res = fonction_requete()
    # Action (.count() ou .collect()) pour forcer Spark à exécuter le calcul (Évaluation Paresseuse / Lazy Evaluation)
    lignes = df_res.count() 
    fin = time.time()
    duree = round(fin - debut, 2)
    results[nom_etape] = duree
    print(f"✅ Terminé en {duree} secondes ({lignes} lignes générées)\n")
    return df_res


#### <span style="background-color: yellow;">NIVEAU 1 : BASELINE (CSV brut + Joins Classiques + InferSchema)</span>

In [5]:
# Problème : Le CSV ne contient pas de types (il faut utiliser inferSchema, qui lit le fichier 2 fois)
# + Les joins provoquent un Shuffle réseau massif entre les 2 workers.
def run_baseline():
    df_flights = spark.read.csv("hdfs://namenode:9000/data/flights", header=True, inferSchema=True)
    df_airports = spark.read.csv("hdfs://namenode:9000/data/airports", header=True, inferSchema=True)
    df_carriers = spark.read.csv("hdfs://namenode:9000/data/carriers", header=True, inferSchema=True)
    
    # Requête analytique : Retard moyen par ville d'origine et compagnie aérienne
    res = df_flights.filter(F.col("Cancelled") == 0) \
        .join(df_airports, df_flights.origin_id == df_airports.airport_id) \
        .join(df_carriers, df_flights.carrier_id == df_carriers.carrier_id) \
        .groupBy(df_airports.ville, df_carriers.nom) \
        .agg(
            F.count("*").alias("total_vols"),
            F.avg("ArrDelay").alias("retard_moyen_arrivee")
        )
    return res

In [6]:
mesurer_requete("1. Référence (CSV + SortMergeJoin)", run_baseline)

--- Exécution : 1. Référence (CSV + SortMergeJoin) ---
✅ Terminé en 56.64 secondes (3752980 lignes générées)



DataFrame[ville: string, nom: string, total_vols: bigint, retard_moyen_arrivee: double]

####  <span style="background-color: yellow;">NIVEAU 2 : OPTIMISATION STOCKAGE (Conversion en PARQUET)</span>

In [7]:
# Optimisation : Le format Parquet est orienté colonnes, compressé (Snappy), et intègre le schéma.
# Spark ne lit que les colonnes nécessaires (Column Pruning) au lieu des 12 colonnes.

print("Conversion des données en format Parquet dans HDFS...")
spark.read.csv("hdfs://namenode:9000/data/flights", header=True, inferSchema=True) \
    .write.mode("overwrite").parquet("hdfs://namenode:9000/data_parquet/flights")
spark.read.csv("hdfs://namenode:9000/data/airports", header=True, inferSchema=True) \
    .write.mode("overwrite").parquet("hdfs://namenode:9000/data_parquet/airports")
spark.read.csv("hdfs://namenode:9000/data/carriers", header=True, inferSchema=True) \
    .write.mode("overwrite").parquet("hdfs://namenode:9000/data_parquet/carriers")

Conversion des données en format Parquet dans HDFS...


In [8]:
def run_parquet():
    df_flights = spark.read.parquet("hdfs://namenode:9000/data_parquet/flights")
    df_airports = spark.read.parquet("hdfs://namenode:9000/data_parquet/airports")
    df_carriers = spark.read.parquet("hdfs://namenode:9000/data_parquet/carriers")
    
    res = df_flights.filter(F.col("Cancelled") == 0) \
        .join(df_airports, df_flights.origin_id == df_airports.airport_id) \
        .join(df_carriers, df_flights.carrier_id == df_carriers.carrier_id) \
        .groupBy(df_airports.ville, df_carriers.nom) \
        .agg(
            F.count("*").alias("total_vols"),
            F.avg("ArrDelay").alias("retard_moyen_arrivee")
        )
    return res

mesurer_requete("2. Format Parquet (Column Pruning)", run_parquet)

--- Exécution : 2. Format Parquet (Column Pruning) ---
✅ Terminé en 15.25 secondes (3752980 lignes générées)



DataFrame[ville: string, nom: string, total_vols: bigint, retard_moyen_arrivee: double]

#### <span style="background-color: yellow;"> NIVEAU 3 : BROADCAST JOIN (Élimination du Shuffle)</span>

In [9]:
# Optimisation : Les tables airports (3400 lignes) et carriers (1500 lignes) sont très petites.
# Au lieu de déplacer les 10M de vols sur le réseau (Shuffle), on copie les petites tables dans la RAM de chaque Worker.
def run_broadcast():
    df_flights = spark.read.parquet("hdfs://namenode:9000/data_parquet/flights")
    df_airports = spark.read.parquet("hdfs://namenode:9000/data_parquet/airports")
    df_carriers = spark.read.parquet("hdfs://namenode:9000/data_parquet/carriers")
    
    res = df_flights.filter(F.col("Cancelled") == 0) \
        .join(F.broadcast(df_airports), df_flights.origin_id == df_airports.airport_id) \
        .join(F.broadcast(df_carriers), df_flights.carrier_id == df_carriers.carrier_id) \
        .groupBy(df_airports.ville, df_carriers.nom) \
        .agg(
            F.count("*").alias("total_vols"),
            F.avg("ArrDelay").alias("retard_moyen_arrivee")
        )
    return res

mesurer_requete("3. Parquet + Broadcast Join", run_broadcast)

--- Exécution : 3. Parquet + Broadcast Join ---
✅ Terminé en 12.25 secondes (3752980 lignes générées)



DataFrame[ville: string, nom: string, total_vols: bigint, retard_moyen_arrivee: double]

 #### <span style="background-color: yellow;">NIVEAU 4 : TUNING PARTITIONS & CACHING (Gestion Mémoire/CPU)</span>

In [11]:
# Optimisation : Par défaut, Spark utilise 200 partitions après un groupBy (spark.sql.shuffle.partitions=200).
# Sur un petit cluster (2 cores au total), 200 tâches créent un énorme overhead de scheduling.
# On réduit à 8 partitions (4x le nombre de cores) et on utilise le Cache en RAM.
def run_tuned():
    spark.conf.set("spark.sql.shuffle.partitions", "8")
    
    df_flights = spark.read.parquet("hdfs://namenode:9000/data_parquet/flights") \
        .select("origin_id", "carrier_id", "ArrDelay", "Cancelled") \
        .filter(F.col("Cancelled") == 0) \
        .cache() # Mise en mémoire RAM
        
    df_airports = spark.read.parquet("hdfs://namenode:9000/data_parquet/airports")
    df_carriers = spark.read.parquet("hdfs://namenode:9000/data_parquet/carriers")
    
    # Premier passage pour charger le cache, puis mesure sur le traitement
    res = df_flights \
        .join(F.broadcast(df_airports), df_flights.origin_id == df_airports.airport_id) \
        .join(F.broadcast(df_carriers), df_flights.carrier_id == df_carriers.carrier_id) \
        .groupBy(df_airports.ville, df_carriers.nom) \
        .agg(
            F.count("*").alias("total_vols"),
            F.avg("ArrDelay").alias("retard_moyen_arrivee")
        )
    return res

In [12]:
mesurer_requete("4. Parquet + Broadcast + Tuning Partitions/Cache", run_tuned)

--- Exécution : 4. Parquet + Broadcast + Tuning Partitions/Cache ---
✅ Terminé en 26.48 secondes (3752980 lignes générées)



DataFrame[ville: string, nom: string, total_vols: bigint, retard_moyen_arrivee: double]

#### <span style="background-color: yellow; color:red">AFFICHAGE DU RÉSUMÉ DU BENCHMARK</span>

In [14]:
print("==========================================")
print("📊 RÉSULTATS FINAUX DU BENCHMARK :")
for etape, temps in results.items():
    print(f"- {etape} : {temps} s")
print("==========================================")

📊 RÉSULTATS FINAUX DU BENCHMARK :
- 1. Référence (CSV + SortMergeJoin) : 56.64 s
- 2. Format Parquet (Column Pruning) : 15.25 s
- 3. Parquet + Broadcast Join : 12.25 s
- 4. Parquet + Broadcast + Tuning Partitions/Cache : 26.48 s


In [15]:
spark.stop()

In [16]:
import json

with open("/home/jovyan/work/benchmark_results.json", "w") as f:
    json.dump(results, f, indent=4)

print("✅ Résultats N1-N4 sauvegardés dans benchmark_results.json !")

✅ Résultats N1-N4 sauvegardés dans benchmark_results.json !
